In [9]:
#!pip install langchain-core
#!pip install langchain-openai

In [35]:
# define the data model with Pydantic to strictly define the allowed categories as outlined in your prompt
from typing import Literal # enforces strict field to exactly one or more specific, predefined values
from pydantic.v1 import BaseModel, Field #date validation lib pyndatic


In [36]:
class Email_route(BaseModel):
    """ channel the right email to the appropriate department based on template"""
    category: Literal["Technical support", "Billing", "General Feedback"] = Field(description="The assigned email to the department that best suited the incoming email.")
    

In [37]:
# create router chain by binding pyndamic to the model so that LLm will always return valid category
from langchain_core.prompts import ChatPromptTemplate # combines structured chat prompts with Pydantic schemas
from langchain_openai import ChatOpenAI # framework used to connect to and interact with OpenAI's conversational language models (like GPT-4o)

In [38]:
#import open API keys
import os, json
credentials = {}

try:
    with open('credentials.json') as file:
        credentials = json.load(file)
        os.environ["OPENAI_API_KEY"] = credentials['OPENAI_API_KEY']
        print("API ready to be used")
except FileNotFoundError:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")



API ready to be used


In [39]:
# initialize LLm
llm = ChatOpenAI(model="gpt-4o", temperature=0) # choose most closely related word in your responce with temp 0
# force llm to return structured data format 
structured_llm = llm.with_structured_output(Email_route) # email route class
# design precise routing prompt 
system_prompt = """
You are an automated email routing system.
When you recieve an email, you will clssify it according to one of the following categories
- Technical Support: Bugs, crashes, errors, and technical issues.
- Billing: Invoices, payments, refunds, and subscription questions.
- Feedback: Suggestions, compliments, and general inquiries.
Be precise and base your classification based on these text provided 
"""
# define prompt 
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{email_content}")
])
# contruct runnable chain
router_chain = prompt | structured_llm

C:\Users\anaconda\Lib\site-packages\langchain_openai\chat_models\base.py:2380: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


In [40]:
# test the routing classification logic by invoking with incoming customer eamil
# email from user 
incoming_user_email = "Hi there, I noticed my credit card was charged twice this month. Can I get a refund?"
result = router_chain.invoke({"email_content": incoming_user_email})
# access the structured email 
print("Route to: {result.category}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [42]:
from typing import Dict, Any
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.language_models import BaseLanguageModel

In [55]:
# Option 2

class Email_routing_classification(BaseModel):
    """ channel the right email to the appropriate department based on template"""
    category: Literal["Technical support", "Billing", "General Feedback"] = Field(description="The assigned category based on the incoming email.")
    department: Literal["Support Department", "Billing Department", "General Department"] = Field(description="Must be 'tech_support', 'billing', or 'finance'")
    confidence: float = Field(..., ge=0.0, le=1.0)
    routed_response: str

    # define prompt template 
    # classification template must contain word classify, email content, and format instructions
    Classification_prompt_template = """
        You are an email routing assitant
        Your task is to classify incoming emails into one of the 3 departments 
    
        Requirements:
        1) Carefully analyse and classify the following email text
        2) Provide your output strictly adhering to the JSON schema below.
    
        Email Content:
        {email_content}
    
        Format Instructions:
        {format_instructions}
    
    """
    # Department Response Prompts
    # Each must include its specific role phrase and the email content
    Tech_support_prompt_template = """
        You are a technical support assitant
        Review the following email and draft a helpful technical response addressing the user's issues.
    
        Email Content:
        {email_content}
    
    """
    
    Billing_prompt_template = """
        You are a billing specialist
        Review the following email and draft a helpful response regarding invoices, charges, or account status..
    
        Email Content:
        {email_content}
    
    """
    General_feedback_prompt_template = """
        You are a customer service representative
        Review the following email and draft a polite response a polite response acknowledging the customer's feedback
    
        Email Content:
        {email_content}
    
    """

# define department handler function
def handle_billing(inputs: Dict[str, Any])-> str:
    return f"Forwarding to billing department. Context: {inputs.get('email', '')}"

def handle_support(inputs: Dict[str, Any])-> str:
    return f"Forwarding to support department. Context: {inputs.get('email', '')}"

def handle_general(inputs: Dict[str, Any])-> str:
    return f"Forwarding to general department. Context: {inputs.get('email', '')}"

# define email classification and routing tool
def classify_route_email(email_content: str, mock_llm_func: callable) -> Email_routing_classification:
    parser = PydanticOutputParser(pydantic_object=Email_routing_classification)
    format_instructions = parser.get_format_instructions()

# Create a prompt that enforces the JSON output format required by the parser
    classified_prompt = PromptTemplate(
        template=Classification_prompt_template,
        input_variables=["email_content", "format_instructions"]
    )
    format_class_prompt = classified_prompt.format(
        email_content = email_content,
        format_instructions = format_instructions
    )
    
    # invoke llm mock function and parse structured data
    llm_output = mock_llm_func.invoke(format_class_prompt)
    
    parsed_result: Email_routing_classification = parser.parse(llm_output.content)
    
    # create internal classification chain 
    classifier_chain = classified_prompt | llm | parser
    
    # Create the RunnableBranch for routing based on the parser's output object
    branch = RunnableBranch(
           ( lambda x: x["classification"].category == "billing", 
            RunnableLambda(handle_billing)
        ),
        RunnableBranch(
          lambda x: x["classification"].category == "technical", 
        RunnableLambda(handle_support)  
        ),
        RunnableLambda(handle_general)
    )
    
    # Combine classification and branching into a unified routing pipeline
    #The dictionary input maps the output of the classification chain to the branch keys

    
    final_router = (
            {"classification": classifier_chain, "email": lambda x: x["email"]}
            | branch
        )
        
    return final_router
